# 04 GridMET + Terrain EDA

This notebook explores the current wildfire severity modeling table:

```text
data/processed/calfire_with_gridmet_terrain.csv
```

Main goals:

1. Check data quality and missingness.
2. Understand the target distribution.
3. Explore wildfire size by year, month, and location.
4. Compare gridMET weather/fire-danger variables across severity tiers.
5. Compare terrain variables across severity tiers.
6. Check feature redundancy before modeling.
7. Check whether a time-based train/test split has distribution shift.

This is **EDA only**. No modeling yet.


## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

INPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain.csv"
EDA_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
EDA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("EDA output directory:", EDA_OUTPUT_DIR)


## 1. Load data

In [ ]:
df = pd.read_csv(INPUT_PATH)

print("Shape:", df.shape)
display(df.head())
display(df.tail())


## 2. Basic data audit

This checks the most important structural issues before any plotting:

- missing required columns
- duplicate IDs
- invalid coordinates
- missing target labels
- missing terrain features


In [ ]:
required_cols = [
    "gridmet_id",
    "Name",
    "Latitude",
    "Longitude",
    "AcresBurned",
    "log_acres",
    "nwcg_size_class",
    "severity_tier",
    "year",
    "month",
]

missing_required = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing_required)

print("\nShape:", df.shape)
print("Duplicate gridmet_id rows:", df["gridmet_id"].duplicated().sum() if "gridmet_id" in df.columns else "gridmet_id missing")

if "Latitude" in df.columns and "Longitude" in df.columns:
    print("\nLatitude range:", df["Latitude"].min(), "to", df["Latitude"].max())
    print("Longitude range:", df["Longitude"].min(), "to", df["Longitude"].max())

if "AcresBurned" in df.columns:
    print("\nAcresBurned missing:", df["AcresBurned"].isna().sum())
    print("AcresBurned <= 0:", (df["AcresBurned"] <= 0).sum())

if "severity_tier" in df.columns:
    print("\nSeverity tier missing:", df["severity_tier"].isna().sum())
    print(df["severity_tier"].value_counts(dropna=False))


In [ ]:
missingness = df.isna().mean().sort_values(ascending=False).to_frame("missing_fraction")
missingness["missing_count"] = df.isna().sum()
display(missingness.head(30))

missingness.to_csv(EDA_OUTPUT_DIR / "missingness_summary.csv")


## 3. Define feature groups

These lists make the rest of the EDA cleaner.

The exact names may vary slightly depending on how your prior notebook named columns, so this cell uses safe checks.


In [ ]:
tier_order = ["Small", "Medium", "Large", "Extreme"]

if "severity_tier" in df.columns:
    df["severity_tier"] = pd.Categorical(df["severity_tier"], categories=tier_order, ordered=True)

target_cols = ["AcresBurned", "log_acres", "nwcg_size_class", "severity_tier"]

terrain_cols = [
    c for c in ["elevation", "slope_degrees", "aspect_degrees", "northness", "eastness"]
    if c in df.columns
]

weather_focus_cols = [
    c for c in [
        "tmmx_K_7d_mean",
        "tmmn_K_7d_mean",
        "vpd_kPa_7d_mean",
        "vs_m_s_7d_mean",
        "rmin_pct_7d_mean",
        "pr_mm_30d_sum",
        "erc_7d_mean",
        "bi_7d_mean",
        "fm100_pct_7d_mean",
        "fm1000_pct_30d_mean",
        "pet_mm_30d_sum",
    ]
    if c in df.columns
]

# Optional Fahrenheit conversions for readability in EDA.
if "tmmx_K_7d_mean" in df.columns:
    df["tmmx_F_7d_mean"] = (df["tmmx_K_7d_mean"] - 273.15) * 9 / 5 + 32
if "tmmn_K_7d_mean" in df.columns:
    df["tmmn_F_7d_mean"] = (df["tmmn_K_7d_mean"] - 273.15) * 9 / 5 + 32

readable_weather_cols = [
    c for c in [
        "tmmx_F_7d_mean",
        "tmmn_F_7d_mean",
        "vpd_kPa_7d_mean",
        "vs_m_s_7d_mean",
        "rmin_pct_7d_mean",
        "pr_mm_30d_sum",
        "erc_7d_mean",
        "bi_7d_mean",
        "fm100_pct_7d_mean",
        "fm1000_pct_30d_mean",
    ]
    if c in df.columns
]

all_focus_cols = readable_weather_cols + terrain_cols

print("Terrain columns:", terrain_cols)
print("Weather/fire-danger columns:", readable_weather_cols)
print("Focus columns:", all_focus_cols)


## 4. Target distribution

This section checks the size labels.

The project uses NWCG size classes and a collapsed modeling tier:

```text
Small:   <100 acres
Medium:  100 to <1,000 acres
Large:   1,000 to <5,000 acres
Extreme: >=5,000 acres
```


In [ ]:
if "nwcg_size_class" in df.columns:
    nwcg_counts = df["nwcg_size_class"].value_counts().sort_index()
    display(nwcg_counts.to_frame("count"))

if "severity_tier" in df.columns:
    tier_counts = df["severity_tier"].value_counts().reindex(tier_order)
    display(tier_counts.to_frame("count"))

    plt.figure(figsize=(8, 5))
    tier_counts.plot(kind="bar")
    plt.title("Fire count by severity tier")
    plt.xlabel("Severity tier")
    plt.ylabel("Number of fires")
    plt.tight_layout()
    plt.show()


In [ ]:
if "AcresBurned" in df.columns:
    display(df["AcresBurned"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame().T)

    plt.figure(figsize=(8, 5))
    df["AcresBurned"].plot(kind="hist", bins=50)
    plt.title("Distribution of acres burned")
    plt.xlabel("Acres burned")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

if "log_acres" in df.columns:
    plt.figure(figsize=(8, 5))
    df["log_acres"].plot(kind="hist", bins=50)
    plt.title("Distribution of log acres burned")
    plt.xlabel("log1p(AcresBurned)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:
if "severity_tier" in df.columns and "AcresBurned" in df.columns:
    tier_acres_summary = df.groupby("severity_tier", observed=True)["AcresBurned"].describe()
    display(tier_acres_summary)

    plt.figure(figsize=(8, 5))
    df.boxplot(column="log_acres", by="severity_tier")
    plt.title("Log acres burned by severity tier")
    plt.suptitle("")
    plt.xlabel("Severity tier")
    plt.ylabel("log1p(AcresBurned)")
    plt.tight_layout()
    plt.show()


## 5. Temporal EDA

This checks whether certain years/months dominate the dataset.

This matters because a time-based validation split may be harder if the training and test periods have very different severity distributions.


In [ ]:
if "year" in df.columns:
    year_counts = df["year"].value_counts().sort_index()
    display(year_counts.to_frame("count"))

    plt.figure(figsize=(10, 5))
    year_counts.plot(kind="bar")
    plt.title("Fire count by year")
    plt.xlabel("Year")
    plt.ylabel("Number of fires")
    plt.tight_layout()
    plt.show()


In [ ]:
if "month" in df.columns:
    month_counts = df["month"].value_counts().sort_index()
    display(month_counts.to_frame("count"))

    plt.figure(figsize=(10, 5))
    month_counts.plot(kind="bar")
    plt.title("Fire count by month")
    plt.xlabel("Month")
    plt.ylabel("Number of fires")
    plt.tight_layout()
    plt.show()


In [ ]:
if "year" in df.columns and "severity_tier" in df.columns:
    yearly_tiers = pd.crosstab(df["year"], df["severity_tier"])
    display(yearly_tiers)

    plt.figure(figsize=(11, 6))
    yearly_tiers.plot(kind="bar", stacked=True, ax=plt.gca())
    plt.title("Severity tier counts by year")
    plt.xlabel("Year")
    plt.ylabel("Number of fires")
    plt.tight_layout()
    plt.show()

    yearly_tier_pct = pd.crosstab(df["year"], df["severity_tier"], normalize="index").round(3)
    display(yearly_tier_pct)


In [ ]:
if "month" in df.columns and "severity_tier" in df.columns:
    monthly_tiers = pd.crosstab(df["month"], df["severity_tier"])
    display(monthly_tiers)

    plt.figure(figsize=(11, 6))
    monthly_tiers.plot(kind="bar", stacked=True, ax=plt.gca())
    plt.title("Severity tier counts by month")
    plt.xlabel("Month")
    plt.ylabel("Number of fires")
    plt.tight_layout()
    plt.show()

    monthly_tier_pct = pd.crosstab(df["month"], df["severity_tier"], normalize="index").round(3)
    display(monthly_tier_pct)


## 6. Spatial EDA

This section checks whether larger fires are geographically clustered.

The scatter plots are not formal maps, but they are useful for a first pass.


In [ ]:
if {"Longitude", "Latitude", "severity_tier"}.issubset(df.columns):
    plt.figure(figsize=(8, 8))
    for tier in tier_order:
        sub = df[df["severity_tier"] == tier]
        if len(sub) == 0:
            continue
        plt.scatter(sub["Longitude"], sub["Latitude"], s=12, alpha=0.6, label=tier)

    plt.title("Wildfire locations by severity tier")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if {"Longitude", "Latitude", "log_acres"}.issubset(df.columns):
    plt.figure(figsize=(8, 8))
    scatter = plt.scatter(df["Longitude"], df["Latitude"], c=df["log_acres"], s=12, alpha=0.7)
    plt.title("Wildfire locations colored by log acres")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.colorbar(scatter, label="log1p(AcresBurned)")
    plt.tight_layout()
    plt.show()


## 7. Weather and fire-danger variables by severity tier

These are the most important gridMET variables to inspect.

Expected general pattern:

- Larger fires may have higher VPD, ERC, BI, temperature, and wind.
- Larger fires may have lower relative humidity and lower fuel moisture.
- These patterns may be noisy because final fire size also depends on terrain, fuels, suppression, ignition, and chance.


In [ ]:
if "severity_tier" in df.columns and readable_weather_cols:
    weather_by_tier_median = df.groupby("severity_tier", observed=True)[readable_weather_cols].median().round(3)
    display(weather_by_tier_median)

    weather_by_tier_mean = df.groupby("severity_tier", observed=True)[readable_weather_cols].mean().round(3)
    display(weather_by_tier_mean)

    weather_by_tier_median.to_csv(EDA_OUTPUT_DIR / "weather_by_tier_median.csv")


In [ ]:
for col in readable_weather_cols:
    plt.figure(figsize=(8, 5))
    df.boxplot(column=col, by="severity_tier")
    plt.title(f"{col} by severity tier")
    plt.suptitle("")
    plt.xlabel("Severity tier")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 8. Terrain variables by severity tier

This checks whether elevation, slope, and aspect-derived variables differ across wildfire severity tiers.


In [ ]:
if "severity_tier" in df.columns and terrain_cols:
    terrain_by_tier_median = df.groupby("severity_tier", observed=True)[terrain_cols].median().round(3)
    display(terrain_by_tier_median)

    terrain_by_tier_mean = df.groupby("severity_tier", observed=True)[terrain_cols].mean().round(3)
    display(terrain_by_tier_mean)

    terrain_by_tier_median.to_csv(EDA_OUTPUT_DIR / "terrain_by_tier_median.csv")


In [ ]:
for col in terrain_cols:
    plt.figure(figsize=(8, 5))
    df.boxplot(column=col, by="severity_tier")
    plt.title(f"{col} by severity tier")
    plt.suptitle("")
    plt.xlabel("Severity tier")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()


## 9. Correlation and redundancy check

Many gridMET variables are related to each other. This section helps identify redundant features before modeling.

For example:

- `erc` and `bi` may be highly correlated.
- `vpd`, `rmin`, and temperature may be related.
- 7-day, 14-day, and 30-day windows of the same variable may be redundant.


In [ ]:
numeric_focus_cols = [c for c in all_focus_cols if c in df.columns]
corr = df[numeric_focus_cols].corr()

display(corr.round(2))

plt.figure(figsize=(12, 10))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation among selected gridMET and terrain features")
plt.tight_layout()
plt.show()

corr.to_csv(EDA_OUTPUT_DIR / "selected_feature_correlation.csv")


In [ ]:
# List highly correlated feature pairs.
threshold = 0.85
pairs = []

for i, col1 in enumerate(corr.columns):
    for j, col2 in enumerate(corr.columns):
        if j <= i:
            continue
        value = corr.loc[col1, col2]
        if pd.notna(value) and abs(value) >= threshold:
            pairs.append((col1, col2, value))

high_corr = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "correlation"]).sort_values(
    "correlation",
    key=lambda s: s.abs(),
    ascending=False,
)

display(high_corr)
high_corr.to_csv(EDA_OUTPUT_DIR / "high_correlation_pairs.csv", index=False)


## 10. Train/test split shift check

Before modeling, compare a likely time split:

```text
Train: 2016 to 2021
Test: 2022 to 2024
```

This helps reveal whether the test period is distributionally different.


In [ ]:
if "year" in df.columns:
    df["split_candidate"] = np.where(df["year"] <= 2021, "train_2016_2021", "test_2022_2024")

    split_counts = df["split_candidate"].value_counts()
    display(split_counts.to_frame("count"))

    if "severity_tier" in df.columns:
        split_tier_counts = pd.crosstab(df["split_candidate"], df["severity_tier"])
        split_tier_pct = pd.crosstab(df["split_candidate"], df["severity_tier"], normalize="index").round(3)

        display(split_tier_counts)
        display(split_tier_pct)


In [ ]:
if "split_candidate" in df.columns and all_focus_cols:
    split_summary = df.groupby("split_candidate")[all_focus_cols].median().round(3)
    display(split_summary)

    split_summary.to_csv(EDA_OUTPUT_DIR / "train_test_candidate_median_features.csv")


In [ ]:
if "split_candidate" in df.columns and "log_acres" in df.columns:
    plt.figure(figsize=(8, 5))
    df.boxplot(column="log_acres", by="split_candidate")
    plt.title("Log acres by candidate split")
    plt.suptitle("")
    plt.xlabel("Split")
    plt.ylabel("log1p(AcresBurned)")
    plt.tight_layout()
    plt.show()


## 11. Initial EDA takeaways

Use this section to write observations in plain English.

Suggested questions:

1. Are `Large` and `Extreme` fires clearly different from `Small` and `Medium` fires?
2. Which variables show the strongest monotonic pattern across severity tiers?
3. Are the largest fires concentrated in specific years or months?
4. Does the 2022–2024 test period look different from 2016–2021?
5. Are there variables that are too redundant to all include in the first model?


In [ ]:
# Fill this in after reviewing the tables and plots.

eda_notes = {
    "target_distribution": "",
    "temporal_patterns": "",
    "spatial_patterns": "",
    "weather_fire_danger_patterns": "",
    "terrain_patterns": "",
    "train_test_shift": "",
    "modeling_implications": "",
}

for key, value in eda_notes.items():
    print(f"{key}: {value}")


## 12. Save a compact EDA-ready feature table

This optional file keeps the target, metadata, selected weather/fire-danger variables, and terrain variables.

It is useful for quick modeling experiments later.


In [ ]:
metadata_cols = [
    c for c in [
        "gridmet_id",
        "Name",
        "fire_start_date",
        "year",
        "month",
        "Latitude",
        "Longitude",
        "AcresBurned",
        "log_acres",
        "nwcg_size_class",
        "severity_tier",
    ]
    if c in df.columns
]

eda_feature_cols = metadata_cols + readable_weather_cols + terrain_cols
eda_feature_cols = list(dict.fromkeys(eda_feature_cols))  # remove duplicates while preserving order

eda_model_base = df[eda_feature_cols].copy()

EDA_MODEL_BASE_PATH = PROCESSED_DIR / "eda_model_base_gridmet_terrain.csv"
eda_model_base.to_csv(EDA_MODEL_BASE_PATH, index=False)

print("Saved compact EDA/modeling base:", EDA_MODEL_BASE_PATH)
print("Shape:", eda_model_base.shape)
display(eda_model_base.head())
